## If working in Google Colab, run this first
Run the cell below at the start of every session. It downloads the course files to your Google Drive and keeps them up to date.
When you see "Setup complete", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.**
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/lectures'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Setup complete. You can now run the rest of the notebook.')

# Introduction to Pose Estimation

Pose estimation is a computer vision task that aims to detect and track key points (landmarks) on objects. Pose estimation is typically applied to the human body; however, it can be extended to localize identifiable landmarks in any object or animal. 

These keypoints represent identifiable locations such as:
- joints (e.g., shoulders, elbows, knees)
- extremities (e.g., hands, feet)
- facial landmarks

Pose estimation is widely used in:
- human movement analysis
- sports science
- rehabilitation
- neuroscience
- human-computer interaction
- animation and gaming

# What is Pose Estimation?

Pose estimation models take an image as input and output the coordinates of keypoints.

## Type of Models

There are two main types of models:
### 1. Regression models
- These models receive an image as input and their output is the prediction of the *(x,y)* position of the different keypoints. These models are usually fast but may lack in accuracy. 

### 2. Heatmaps models
- These models receive an image as input, and the output is a set of heatmaps with the likely *(x,y)* position of the different keypoints. 
<div style="text-align:center">
<img src="images/heatmap-based-models.png" alt="plot" width="400" />
</div>

There is one heatmap per keypoint. During training, the target heatmap is a 2D Gaussian centered at the true keypoint position (the width of the Gaussian is fixed). After training, the position of the **peak** of the heatmap is the predicted keypoint position, and the **height** of the peak is the confidence. If a keypoint is hidden (occluded), the heatmap has no clear peak, so the confidence is low.


## Tasks

There are three main task performed by pose estimation models:

### 1. 2D Pose Estimation
- Outputs (x, y) pixel coordinates

<div style="text-align:center">
<img src="images/COCOPose.webp" alt="plot" width="400" />
</div>

### 2. 3D Pose Estimation
- Outputs (x, y, z) coordinates
- Estimated depth information

<div style="text-align:center">
<img src="images/3dpose.gif" alt="plot" width="600" />
</div>

**3D Pose Estimation** can be computed in three different ways (usually from most to least accurate):

a - Use **multiple cameras** and reconstruct the 3D position of each keypoint by triangulation (the same principle as marker-based motion capture).

b - Use a **depth sensor** (for example a Kinect or a phone depth camera) to measure the distance from each keypoint to the camera.

c - Use **one camera** and a model that estimates the depth of each keypoint. This is the easiest to use, but the depth is an estimate, not a measurement.

### 3. Human Mesh Recovery
- These models estimate a full 3D surface (a **mesh**) of the body, not only keypoints. Most of them fit a parametric body model, such as SMPL, to the image.
- The mesh has thousands of vertices. The body model makes sure that the result has a realistic human shape.

<div style="text-align:center">
<img src="images/mesh.png" alt="plot" width="600" />
</div>

## Common Pose Estimation Problems

Pose estimation models can focus on different body parts:

### Full Body Pose
- Detects major joints (e.g., shoulders, hips, knees)
- Most models estimate 17–33 keypoints; however, many models estimate more keypoints. 

### Hand Pose
- Detects the position of different finger joints
- Most models estimate the position of 21 keypoints per hand (finger joints + tips and wrist). 

### Face Pose
- Detects facial landmarks. (eyes, nose, mouth, jawline)
- Common sets have 5, 68 or 98 keypoints around the eyes, nose, mouth and jawline. Some models (for example MediaPipe Face Mesh) estimate 468 keypoints.

### Whole Body Pose
- These models combine body, hands, and face.
- Used in detailed motion analysis and animation

## Pose Estimation Datasets

Pose estimation models are trained on labeled datasets where keypoints are manually annotated.

### Common Datasets

#### COCO Pose (Common Objects in Context)
- 17 keypoints for human body
- One of the most widely used datasets

#### MPII Human Pose
- 16 keypoints; images of people doing many everyday activities
<div style="text-align:center">
<img src="images/MPII-keypoints.png" alt="plot" width="200" />
</div>

#### Human3.6M
- Large-scale dataset with 3D ground truth
- Captured in controlled lab environments with actors performing multiple movements. Data was captured with an array of cameras. 

<div style="text-align:center">
<img src="images/Human3-6.png" alt="plot" width="600" />
</div>

#### OpenPose keypoint format (BODY_25)
- OpenPose is a **model**, not a dataset. It was trained on COCO and MPII, and it uses its own 25-keypoint format (BODY_25), which adds the feet. You will see this format in notebook 08.02 part 2.
<div style="text-align:center">
<img src="images/OpenPose.jpeg" alt="plot" width="200" />
</div>

#### MediaPipe training data (not public)
- Google trained the MediaPipe models on its own data, which is not public. Part of the data is synthetic (3D avatars).
- Annotations for the body, face and hands
<div style="text-align:center">
<img src="images/mediapipe.png" alt="plot" width="200" />
</div>

### other popular datasets

- AI Challenger
- CrowdPose
- sub-JHMDB
- Halpe
- PoseTrack18

---

It is important to understand that the dataset defines:
- which keypoints are detected
- how accurate the model can be
- how well the model generalizes

---

## Popular Pose Estimation Models

Some models are designed for specific use cases:

### YOLO Pose
- Detects 2D keypoints
- Designed for high processing speed while maintaining high accuracy. 
- Trained on the COCO Pose dataset, but can be fine-tuned with any other dataset. 

### MediaPipe
- Detects 33 body keypoints, with an estimated depth for each keypoint
- Designed for high processing speed, also on phones
- The code is open source, but the training data is not public, so the models are difficult to retrain with your own data

#### MediaPipe Pose
- It is a lightweight and accurate model that Detects 3D body keypoints
<div style="text-align:center">
<img src="images/mediapipePose.png" alt="plot" width="400" />
</div>

#### MediaPipeHands
 - It is a lightweight and accurate model that Detects 3D hand keypoints

<div style="text-align:center">
<img src="images/mediapipeHands.jpg" alt="plot" width="400" />
</div>

#### MediaPipe Face
- It is a lightweight and accurate model that Detects 3D face keypoints
<div style="text-align:center">
<img src="images/mediapipeFace.gif" alt="plot" width="200" />
</div>

### HRNet
HRNet is a family of accurate pose estimation models. Most CNNs make the image smaller and smaller in each layer, and lose spatial detail. HRNet keeps a **high-resolution branch** through the whole network, in parallel with lower-resolution branches, and combines them. This gives more precise keypoints. (The input image is still resized, for example to 256 × 192 pixels.)

HRNet models are available in the MMPose library.

### Whole Body
- OpenPose and AlphaPose are popular models that combine the body, hands and face.

<div style="text-align:center">
<img src="images/fullbody.webp" alt="plot" width="400" />
</div>

### WHAM
- Reconstructs a 3D body mesh from a **video** (one camera). It also estimates how the body moves in the world, and it keeps the motion smooth between frames.

### Animals
- Pose estimation models have also been extended to other applications, including animals. A few databases for animal pose estimation have been developed over the years and multiple models are available in libraries such as MMPose. 

<div style="text-align:center">
<img src="images/animaPose.webp" alt="plot" width="400" />
</div>

## Pose estimation with YOLO

We use the YOLO pose model (`yolo26s-pose.pt`), trained on COCO. It finds each person in the image and gives 17 keypoints for each person. For each keypoint it gives the (x, y) position in pixels and a **confidence** between 0 and 1.

| Index | Keypoint | Index | Keypoint |
| --- | --- | --- | --- |
| 0 | nose | 9, 10 | left, right wrist |
| 1, 2 | left, right eye | 11, 12 | left, right hip |
| 3, 4 | left, right ear | 13, 14 | left, right knee |
| 5, 6 | left, right shoulder | 15, 16 | left, right ankle |
| 7, 8 | left, right elbow | | |

Two practical problems:
1. **More than one person.** The lift videos show several people. We keep the person with the **largest box** (the person closest to the camera), not the most confident one, so that the selected person does not change between frames.
2. **Keypoints that are not detected.** YOLO gives the position (0, 0) for these keypoints, with a low confidence. We replace keypoints with a confidence below 0.3 by NaN, so that they are not plotted.

In [ ]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
import numpy as np
import matplotlib.pyplot as plt
import cv2
from ultralytics import YOLO
from ipywidgets import IntSlider, Play, HBox, VBox, jslink, interactive_output
from IPython.display import display

model = YOLO("../models/yolo26s-pose.pt")   # COCO 17-keypoint pose model

edges = [(5, 7), (7, 9), (6, 8), (8, 10),          # arms
         (5, 6), (5, 11), (6, 12), (11, 12),       # trunk
         (11, 13), (13, 15), (12, 14), (14, 16),   # legs
         (0, 5), (0, 6)]                           # head to shoulders

def run_pose(video_path, min_conf=0.3):
    """Run YOLO pose on every frame. Keep the person with the largest box.
    Returns keypoints (frames, 17, 2) in pixels, with NaN for missing keypoints, and the frame size."""
    results = model.predict(source=video_path, stream=True, verbose=False)
    pose_seq = []
    for r in results:
        kp = np.full((17, 2), np.nan)
        if r.boxes is not None and len(r.boxes) > 0 and r.keypoints is not None:
            b = r.boxes.xyxy.cpu().numpy()
            k = int(np.argmax((b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])))   # largest box
            xy = r.keypoints.xy.cpu().numpy()[k]
            conf = r.keypoints.conf.cpu().numpy()[k]
            kp = np.where(conf[:, None] >= min_conf, xy, np.nan)
        pose_seq.append(kp)
    h, w = r.orig_shape
    return np.array(pose_seq), w, h

def show_pose_player(pose_seq, w, h):
    """Interactive player: a slider and a play button to see the skeleton in each frame."""
    def plot_pose_frame(frame_idx):
        k = pose_seq[frame_idx]
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.scatter(k[:, 0], k[:, 1], s=30, c="red")
        for a, b in edges:
            ax.plot(k[[a, b], 0], k[[a, b], 1], color="tab:cyan", linewidth=2)
        ax.set_xlim(0, w); ax.set_ylim(h, 0)       # image coordinates: y goes down
        ax.set_aspect("equal", adjustable="box")
        ax.set_title(f"Pose keypoints - frame {frame_idx}")
        ax.set_xlabel("X (pixels)"); ax.set_ylabel("Y (pixels)")
        plt.show()
    slider = IntSlider(value=0, min=0, max=len(pose_seq) - 1, step=1, description="Frame")
    play = Play(value=0, min=0, max=len(pose_seq) - 1, step=1, interval=80, description="Press play")
    jslink((play, "value"), (slider, "value"))
    display(VBox([HBox([play, slider]), interactive_output(plot_pose_frame, {"frame_idx": slider})]))

Run the model on the first lift video (about 30 s on a CPU), then use the player to look at the skeleton in each frame.

In [ ]:
pose_one, w, h = run_pose("data/lift_one.mp4")
print('Frames:', pose_one.shape[0], '| frames with a detected hip:', np.isfinite(pose_one[:, 11, 0]).sum())
show_pose_player(pose_one, w, h)

The same for the second lift video (about 1 minute).

In [ ]:
pose_two, w, h = run_pose("data/lift_two.mp4")
print('Frames:', pose_two.shape[0], '| frames with a detected hip:', np.isfinite(pose_two[:, 11, 0]).sum())
show_pose_player(pose_two, w, h)

## From keypoints to a movement signal

The keypoints change from frame to frame, so each keypoint gives a signal. For example, the vertical position of the hips shows the squat. Remember that in image coordinates **y increases downwards**, so we invert the axis.

In [ ]:
cap = cv2.VideoCapture("data/lift_two.mp4"); fps = cap.get(cv2.CAP_PROP_FPS); cap.release()
t = np.arange(pose_two.shape[0]) / fps
hip_y = np.nanmean(pose_two[:, [11, 12], 1], axis=1)       # mean of the left and right hip

plt.figure(figsize=(10, 4))
plt.plot(t, hip_y)
plt.gca().invert_yaxis()                                   # up in the plot = up in the image
plt.xlabel('Time (s)'); plt.ylabel('Hip vertical position (pixels)')
plt.title('Hip height during the lift')
plt.show()

**Questions to think about**
- The signal is in pixels. What would you need to convert it to meters?
- Where could the signal be wrong? (Hint: what happens if another person is larger in the image for a few frames?)